In [ ]:
!pip -q install qdrant-client transformers torch torchvision pillow tqdm


In [ ]:
import os, uuid, time
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm


CSV_PATH = "/content/drive/MyDrive/VectorsinOrbit/last/vio_EDA.csv"
IMG_ROOT = "/content/drive/MyDrive/VectorsinOrbit/last/images/"
EXTS = [".jpg", ".jpeg", ".png", ".webp"]

df = pd.read_csv(CSV_PATH)
df.drop(columns=["Unnamed: 0"], inplace=True)
print("Catalog shape:", df.shape)
df.head(2)


Catalog shape: (42427, 11)


,Product ID,Product Category,Product Subcategory,price_bucket,Product Name,Product Price ($),Product Rating,Total Ratings,Product Bayes_rating,Product Value_score,Product Image
0,0,Electronics,Smartphones,premium,"SAMSUNG Galaxy S25 Ultra Cell Phone, 512GB AI ...",1219.99,4.3,247,4.315942,0.019505,https://m.media-amazon.com/images/I/61n0lmxP5-...
1,1,Electronics,Smartphones,mid,"SAMSUNG Galaxy A16 5G A Series Cell Phone, Unl...",174.99,4.2,166,4.235365,0.123873,https://m.media-amazon.com/images/I/71KGkQ+KOK...


In [ ]:
def qdrant_id_from_product_id(product_id: str) -> str:
    # Stable UUID: same product_id -> same Qdrant ID
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"vio:{product_id}"))

def py_scalar(x):
    if x is None:
        return None
    if isinstance(x, float) and np.isnan(x):
        return None
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if isinstance(x, (pd.Timestamp, np.datetime64)):
        return str(x)
    return x

def sanitize_payload(d):
    return {k: py_scalar(v) for k, v in d.items()}

def resolve_path(pid):
    pid = str(pid)
    for ext in EXTS:
        p = os.path.join(IMG_ROOT, pid + ext)
        if os.path.exists(p):
            return p
    return None

def build_doc_text(r):
    return (
        f"{r['Product Name']} | {r['Product Category']} | {r['Product Subcategory']} | "
        f"bucket:{r['price_bucket']} | price:{r['Product Price ($)']}"
    )

def make_payload(r):
    d = {
        "product_id": str(r["Product ID"]),
        "name": r["Product Name"],
        "category": r["Product Category"],
        "subcategory": r["Product Subcategory"],
        "price": r["Product Price ($)"],
        "price_bucket": r["price_bucket"],
        "rating": r["Product Rating"],
        "total_ratings": r["Total Ratings"],
        "bayes_rating": r["Product Bayes_rating"],
        "value_score": r["Product Value_score"],
        "image_url": r["Product Image"],  # URL only for display
    }
    return sanitize_payload(d)


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
from google.colab import userdata

QDRANT_URL = userdata.get('QdrantURL')
QDRANT_API_KEY = userdata.get('QdrantAPI')

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

COLLECTION = "vio_products"

existing = [c.name for c in client.get_collections().collections]
if COLLECTION in existing:
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config={
        "text": VectorParams(size=512, distance=Distance.COSINE),
        "image": VectorParams(size=512, distance=Distance.COSINE),
    },
)

client.get_collection(COLLECTION)


CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=0, segments_count=2, config=CollectionConfig(params=CollectionParams(vectors={'image': VectorParams(size=512, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), 'text': VectorParams(size=512, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, 

In [ ]:
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)

@torch.no_grad()
def embed_text_batch(texts):
    inputs = processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)
    feats = model.get_text_features(**inputs)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy().astype(np.float32)

@torch.no_grad()
def embed_image_batch(pil_images):
    inputs = processor(images=pil_images, return_tensors="pt").to(device)
    feats = model.get_image_features(**inputs)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy().astype(np.float32)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
from qdrant_client.models import PointStruct

# Stable settings (start here)
ROWS_PER_CHUNK = 128        # 128 is safe; increase to 256 later
UPLOAD_BATCH_SIZE = 32      # safe for Cloud; can try 64 later
IMG_BATCH = 32              # safe on T4; can try 64 later
PARALLEL = 1                # avoid thread crashes in Colab

def upload_with_retries(points, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            client.upload_points(
                collection_name=COLLECTION,
                points=points,
                batch_size=UPLOAD_BATCH_SIZE,
                parallel=PARALLEL,
            )
            return
        except Exception as e:
            if attempt == max_retries:
                raise
            time.sleep(2 * attempt)

n = len(df)

for start in tqdm(range(0, n, ROWS_PER_CHUNK), desc="Ingest"):
    chunk = df.iloc[start:start+ROWS_PER_CHUNK].copy().reset_index(drop=True)

    # text embeddings for all rows
    docs = chunk.apply(build_doc_text, axis=1).tolist()
    text_vecs = embed_text_batch(docs)

    # find images for rows in this chunk
    row_items = []
    for i in range(len(chunk)):
        pid_raw = str(chunk.loc[i, "Product ID"])
        p = resolve_path(pid_raw)
        if p is not None:
            row_items.append((i, pid_raw, p))

    # compute image vecs in mini-batches, map row_index -> img_vec
    row_to_imgvec = {}
    for b in range(0, len(row_items), IMG_BATCH):
        sub = row_items[b:b+IMG_BATCH]
        pil_imgs, row_ids = [], []
        for row_i, pid_raw, path in sub:
            try:
                pil_imgs.append(Image.open(path).convert("RGB"))
                row_ids.append(row_i)
            except:
                pass
        if not pil_imgs:
            continue
        img_vecs = embed_image_batch(pil_imgs)
        for k, row_i in enumerate(row_ids):
            row_to_imgvec[row_i] = img_vecs[k]

    # build points
    points = []
    for i in range(len(chunk)):
        r = chunk.loc[i]
        pid_raw = str(r["Product ID"])
        qid = qdrant_id_from_product_id(pid_raw)

        vectors = {"text": text_vecs[i].tolist()}
        if i in row_to_imgvec:
            vectors["image"] = row_to_imgvec[i].tolist()

        points.append(
            PointStruct(
                id=qid,
                vector=vectors,
                payload=make_payload(r),
            )
        )

    upload_with_retries(points)

print("Done.")


Ingest:   0%|          | 0/332 [00:00<?, ?it/s]

# Upload Missing points on qdrant (just run the cells below)

In [ ]:
!pip -q install qdrant-client transformers torch torchvision pillow tqdm

import os, uuid, time
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct


In [ ]:
CSV_PATH = "/content/drive/MyDrive/VectorsinOrbit/last/vio_EDA.csv"
IMG_ROOT = "/content/drive/MyDrive/VectorsinOrbit/last/images/"
EXTS = [".jpg", ".jpeg", ".png", ".webp"]

df = pd.read_csv(CSV_PATH)
df.drop(columns=["Unnamed: 0"], inplace=True)
df = df[df["Product ID"].notna()].copy()
print("CSV rows:", len(df))
df.head(2)

CSV rows: 42427


,Product ID,Product Category,Product Subcategory,price_bucket,Product Name,Product Price ($),Product Rating,Total Ratings,Product Bayes_rating,Product Value_score,Product Image
0,0,Electronics,Smartphones,premium,"SAMSUNG Galaxy S25 Ultra Cell Phone, 512GB AI ...",1219.99,4.3,247,4.315942,0.019505,https://m.media-amazon.com/images/I/61n0lmxP5-...
1,1,Electronics,Smartphones,mid,"SAMSUNG Galaxy A16 5G A Series Cell Phone, Unl...",174.99,4.2,166,4.235365,0.123873,https://m.media-amazon.com/images/I/71KGkQ+KOK...


In [ ]:
def qdrant_id_from_product_id(product_id: str) -> str:
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"vio:{product_id}"))

def py_scalar(x):
    if x is None:
        return None
    if isinstance(x, float) and np.isnan(x):
        return None
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if isinstance(x, (pd.Timestamp, np.datetime64)):
        return str(x)
    return x

def sanitize_payload(d):
    return {k: py_scalar(v) for k, v in d.items()}

def resolve_path(pid):
    pid = str(pid)
    for ext in EXTS:
        p = os.path.join(IMG_ROOT, pid + ext)
        if os.path.exists(p):
            return p
    return None

def build_doc_text(r):
    return (
        f"{r['Product Name']} | {r['Product Category']} | {r['Product Subcategory']} | "
        f"bucket:{r['price_bucket']} | price:{r['Product Price ($)']}"
    )

def make_payload(r):
    d = {
        "product_id": str(r["Product ID"]),
        "name": r["Product Name"],
        "category": r["Product Category"],
        "subcategory": r["Product Subcategory"],
        "price": r["Product Price ($)"],
        "price_bucket": r["price_bucket"],
        "rating": r["Product Rating"],
        "total_ratings": r["Total Ratings"],
        "bayes_rating": r["Product Bayes_rating"],
        "value_score": r["Product Value_score"],
        "image_url": r["Product Image"],
    }
    return sanitize_payload(d)


In [ ]:
from google.colab import userdata

QDRANT_URL = userdata.get('QdrantURL')
QDRANT_API_KEY = userdata.get('QdrantAPI')
COLLECTION = "vio_products"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
info = client.get_collection(COLLECTION)
print("Qdrant points_count:", info.points_count)

existing_ids = set()
offset = None

while True:
    points, offset = client.scroll(
        collection_name=COLLECTION,
        with_payload=False,
        with_vectors=False,
        limit=1000,
        offset=offset,
    )
    for p in points:
        existing_ids.add(str(p.id))
    if offset is None:
        break

print("Existing IDs fetched:", len(existing_ids))


Qdrant points_count: 18688
Existing IDs fetched: 18688


In [ ]:
df["qid"] = df["Product ID"].astype(str).map(qdrant_id_from_product_id)
missing_df = df[~df["qid"].isin(existing_ids)].copy()

print("Missing rows to upload:", len(missing_df))
missing_df.head(2)


Missing rows to upload: 23739


,Product ID,Product Category,Product Subcategory,price_bucket,Product Name,Product Price ($),Product Rating,Total Ratings,Product Bayes_rating,Product Value_score,Product Image,qid
18688,19362,Groceries,Instant soups,cheap,SOUP SECRETS Lipton Cup-a-Soup Instant Soup Ch...,1.78,4.7,9227,4.699430,24.104372,https://m.media-amazon.com/images/I/81xwyGQaaG...,8f09dcb1-7f21-5412-9b79-b8a17c82129e
18689,19363,Groceries,Instant soups,mid,Knorr Mushroom Soup Mix 1.52oz (3x2 Packs for ...,9.99,4.3,347,4.311645,2.525787,https://m.media-amazon.com/images/I/810NpVDZPU...,f17f7e2c-4944-548a-b9c4-652b4ee5c1d6


In [ ]:
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)

@torch.no_grad()
def embed_text_batch(texts):
    inputs = processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)
    feats = model.get_text_features(**inputs)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy().astype(np.float32)

@torch.no_grad()
def embed_image_batch(pil_images):
    inputs = processor(images=pil_images, return_tensors="pt").to(device)
    feats = model.get_image_features(**inputs)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy().astype(np.float32)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
ROWS_PER_CHUNK = 128        # safe for timeouts; can try 128 later
UPLOAD_BATCH_SIZE = 32     # safe for Cloud; can try 32 later
IMG_BATCH = 32             # safe for T4; can try 32 later
PARALLEL = 1               # most stable in Colab

def upload_with_retries(points, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            client.upload_points(
                collection_name=COLLECTION,
                points=points,
                batch_size=UPLOAD_BATCH_SIZE,
                parallel=PARALLEL,
            )
            return
        except Exception:
            if attempt == max_retries:
                raise
            time.sleep(2 * attempt)

missing_df = missing_df.reset_index(drop=True)
n = len(missing_df)

for start in tqdm(range(0, n, ROWS_PER_CHUNK), desc="Resume ingest"):
    chunk = missing_df.iloc[start:start+ROWS_PER_CHUNK].copy().reset_index(drop=True)

    # --- text embeddings
    docs = chunk.apply(build_doc_text, axis=1).tolist()
    text_vecs = embed_text_batch(docs)

    # --- image embeddings for rows with existing image files
    row_items = []
    for i in range(len(chunk)):
        pid_raw = str(chunk.loc[i, "Product ID"])
        p = resolve_path(pid_raw)
        if p is not None:
            row_items.append((i, p))

    row_to_imgvec = {}
    for b in range(0, len(row_items), IMG_BATCH):
        sub = row_items[b:b+IMG_BATCH]
        pil_imgs, row_ids = [], []
        for row_i, path in sub:
            try:
                pil_imgs.append(Image.open(path).convert("RGB"))
                row_ids.append(row_i)
            except:
                pass
        if not pil_imgs:
            continue

        img_vecs = embed_image_batch(pil_imgs)
        for k, row_i in enumerate(row_ids):
            row_to_imgvec[row_i] = img_vecs[k]

    # --- build points
    points = []
    for i in range(len(chunk)):
        r = chunk.loc[i]
        pid_raw = str(r["Product ID"])
        qid = qdrant_id_from_product_id(pid_raw)

        vectors = {"text": text_vecs[i].tolist()}
        if i in row_to_imgvec:
            vectors["image"] = row_to_imgvec[i].tolist()

        points.append(PointStruct(id=qid, vector=vectors, payload=make_payload(r)))

    upload_with_retries(points)

print("Resume ingestion completed for missing rows.")


Resume ingest:   0%|          | 0/186 [00:00<?, ?it/s]

Resume ingestion completed for missing rows.
